# Module A3 - Face Recognition Fundamentals & Database Setup

This notebook builds the face recognition mapping database for returning customer detection.

It is configured to run either locally (using synthetic fallbacks) or on Kaggle using a subset of LFW (Labeled Faces in the Wild).

In [ ]:
import os
import glob
import shutil
import pickle
import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

In [ ]:
# Auto-detect Kaggle environment
KAGGLING = os.path.exists('/kaggle')

if KAGGLING:
    print("Running in Kaggle environment!")
    # Case-insensitive scan for LFW dataset
    input_dirs = []
    if os.path.exists('/kaggle/input'):
        for d in os.listdir('/kaggle/input'):
            d_lower = d.lower()
            if 'lfw' in d_lower or 'face' in d_lower:
                input_dirs.append(os.path.join('/kaggle/input', d))
                
    if input_dirs:
        LFW_DIR = os.path.join(input_dirs[0], 'lfw-deepfunneled')
    else:
        LFW_DIR = "/kaggle/input/labeled-faces-in-the-wild/lfw-deepfunneled"
    print(f"Kaggle LFW Directory: {LFW_DIR}")
    
    MODEL_DIR = "/kaggle/working/models"
    FACES_DIR = "/kaggle/working/faces"
else:
    print("Running locally.")
    LFW_DIR = None
    FACES_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'faces'))
    MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'app', 'models'))

os.makedirs(FACES_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
def create_synthetic_face(name, file_path):
    img = Image.new('RGB', (150, 150), (220, 220, 220))
    draw = ImageDraw.Draw(img)
    # Head
    draw.ellipse([25, 25, 125, 125], fill=(255, 200, 150), outline=(0, 0, 0), width=1)
    # Eyes
    draw.ellipse([45, 55, 60, 70], fill=(255, 255, 255), outline=(0, 0, 0))
    draw.ellipse([50, 60, 55, 65], fill=(0, 0, 0))
    draw.ellipse([90, 55, 105, 70], fill=(255, 255, 255), outline=(0, 0, 0))
    draw.ellipse([95, 60, 100, 65], fill=(0, 0, 0))
    # Nose
    draw.polygon([(75, 65), (70, 85), (80, 85)], fill=(240, 170, 130))
    # Smile
    draw.arc([50, 85, 100, 110], start=0, end=180, fill=(0, 0, 0), width=2)
    img.save(file_path)

if KAGGLING and os.path.exists(LFW_DIR):
    print("Loading face images from LFW...")
    person_dirs = [d for d in os.listdir(LFW_DIR) if os.path.isdir(os.path.join(LFW_DIR, d))]
    eligible_people = []
    for person in person_dirs:
        person_path = os.path.join(LFW_DIR, person)
        images = [f for f in os.listdir(person_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if len(images) >= 3:
            eligible_people.append((person, images))
            
    # Pick top 5 people by count
    eligible_people = sorted(eligible_people, key=lambda x: len(x[1]), reverse=True)[:5]
    print("Selected LFW profiles:", [p[0] for p in eligible_people])
    
    for person, images in eligible_people:
        dest_person_dir = os.path.join(FACES_DIR, person)
        os.makedirs(dest_person_dir, exist_ok=True)
        for img in images[:3]:
            src_path = os.path.join(LFW_DIR, person, img)
            dest_path = os.path.join(dest_person_dir, img)
            if not os.path.exists(dest_path):
                try:
                    os.symlink(src_path, dest_path)
                except:
                    shutil.copy(src_path, dest_path)
    print("LFW symlinking complete.")
else:
    print("Checking for local face directories...")
    profiles = ['Alice', 'Bob', 'Charlie']
    existing_subdirs = [d for d in os.listdir(FACES_DIR) if os.path.isdir(os.path.join(FACES_DIR, d))]
    
    if not existing_subdirs:
        print("Generating local synthetic fallback faces...")
        for name in profiles:
            path = os.path.join(FACES_DIR, name)
            os.makedirs(path, exist_ok=True)
            for i in range(2):
                create_synthetic_face(name, os.path.join(path, f"{name.lower()}_{i}.jpg"))
        print("Synthetic faces generated!")
    else:
        print("Face directories detected:", existing_subdirs)

In [ ]:
use_dlib = False
try:
    import face_recognition
    use_dlib = True
    print("Successfully imported face_recognition (dlib-based) engine.")
except ImportError:
    print("face_recognition library not found. Falling back to OpenCV LBPH engine.")

In [ ]:
if use_dlib:
    known_encodings = []
    known_names = []

    for person_name in os.listdir(FACES_DIR):
        person_dir = os.path.join(FACES_DIR, person_name)
        if not os.path.isdir(person_dir): continue
        
        for filename in os.listdir(person_dir):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            img_path = os.path.join(person_dir, filename)
            
            image = face_recognition.load_image_file(img_path)
            encodings = face_recognition.face_encodings(image)
            
            if len(encodings) > 0:
                known_encodings.append(encodings[0])
                known_names.append(person_name)
                print(f"Encoded {person_name} from {filename}")
            else:
                dummy_encoding = np.random.normal(size=128)
                dummy_encoding = dummy_encoding / np.linalg.norm(dummy_encoding)
                known_encodings.append(dummy_encoding)
                known_names.append(person_name)
                print(f"Warning: No face found in {filename}. Created a mock embedding vector.")
                
    face_db_path = os.path.join(MODEL_DIR, 'face_db.pkl')
    with open(face_db_path, 'wb') as f:
        pickle.dump({"encodings": known_encodings, "names": known_names, "engine": "face_recognition"}, f)
    print(f"Database saved to: {face_db_path}")

In [ ]:
if not use_dlib:
    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    face_cascade = cv2.CascadeClassifier(cascade_path)
    
    label_map = {}
    current_id = 0
    
    faces_data = []
    labels_data = []
    
    for person_name in os.listdir(FACES_DIR):
        person_dir = os.path.join(FACES_DIR, person_name)
        if not os.path.isdir(person_dir): continue
        
        if person_name not in label_map:
            label_map[current_id] = person_name
            current_id += 1
            
        person_label = [k for k, v in label_map.items() if v == person_name][0]
        
        for filename in os.listdir(person_dir):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            img_path = os.path.join(person_dir, filename)
            
            gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=2)
            
            if len(faces) > 0:
                for (x, y, w, h) in faces:
                    face_roi = gray[y:y+h, x:x+w]
                    face_roi = cv2.resize(face_roi, (100, 100))
                    faces_data.append(face_roi)
                    labels_data.append(person_label)
                    print(f"Detected face for {person_name} in {filename}")
            else:
                face_roi = cv2.resize(gray, (100, 100))
                faces_data.append(face_roi)
                labels_data.append(person_label)
                print(f"Warning: Cascade missed face in {filename}. Using cropped image fallback.")

    try:
        recognizer = cv2.face.LBPHFaceRecognizer_create()
        recognizer.train(faces_data, np.array(labels_data))
        
        model_xml_path = os.path.join(MODEL_DIR, 'lbph_recognizer.xml')
        recognizer.write(model_xml_path)
        
        face_db_path = os.path.join(MODEL_DIR, 'face_db.pkl')
        with open(face_db_path, 'wb') as f:
            pickle.dump({"label_map": label_map, "xml_path": model_xml_path, "engine": "lbph"}, f)
            
        print(f"LBPH Model written to {model_xml_path}")
        print(f"Face Database metadata saved to: {face_db_path}")
    except AttributeError:
        print("\nError: 'cv2.face' module is not installed. Writing simulated database metadata.")
        face_db_path = os.path.join(MODEL_DIR, 'face_db.pkl')
        with open(face_db_path, 'wb') as f:
            pickle.dump({"label_map": label_map, "xml_path": None, "engine": "simulated"}, f)
        print(f"Simulated database metadata saved to: {face_db_path}")

In [ ]:
with open(os.path.join(MODEL_DIR, 'face_db.pkl'), 'rb') as f:
    db = pickle.load(f)
print("Loaded Database Metadata successfully:")
for key, val in db.items():
    if key == 'encodings':
        print(f" - {key}: List of {len(val)} embeddings")
    else:
        print(f" - {key}: {val}")